# Employee Attrition Prediction + Retention Simulator

**Problem:** Predict which employees are likely to leave so HR can act early.

**Dataset:** IBM HR Analytics Employee Attrition (Kaggle, 1,470 rows, synthetic).

**What makes this project different**
1. **Cost-based decision threshold**: the cut-off is chosen by maximising estimated rupee savings, not by a generic F1.
2. **Calibrated probabilities**: "35% risk" really means about 35%.
3. **Retention What-If Simulator**: for any employee, the model tests HR actions (remove overtime, raise pay, add stock options...) and ranks them by risk reduction.
4. **HR Action Report**: top at-risk current employees with their main risk drivers and best action.
5. **Fairness audit** and an interactive Gradio app.

**How to run:** Runtime -> Run all. Upload the Kaggle CSV when asked (Cell 3).

## 1. Setup

In [ ]:
!pip install -q imbalanced-learn shap gradio

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib, shap, xgboost as xgb
from google.colab import files

from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate,
                                     cross_val_predict, RandomizedSearchCV)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix, RocCurveDisplay)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

SEED = 42
sns.set_theme(style="whitegrid")

# ---- Business assumptions (edit these, they drive the decision threshold) ----
COST_REPLACE      = 500_000   # Rs: cost of losing + replacing one employee
COST_INTERVENTION = 50_000    # Rs: cost of one retention action (raise, bonus, coaching...)
SUCCESS_RATE      = 0.40      # share of flagged leavers an intervention actually retains

## 2. Load data

In [ ]:
uploaded = files.upload()                       # choose WA_Fn-UseC_-HR-Employee-Attrition.csv
raw = pd.read_csv(list(uploaded.keys())[0])
print("Shape:", raw.shape, "| Missing values:", int(raw.isnull().sum().sum()))
display(raw.head())

## 3. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
(raw['Attrition'].value_counts(normalize=True) * 100).plot(kind='bar', ax=axes[0, 0], title='Attrition share (%)')
for ax, col in zip(axes.flat[1:], ['OverTime', 'MaritalStatus', 'BusinessTravel', 'Department', 'JobRole']):
    rate = raw.groupby(col)['Attrition'].apply(lambda s: (s == 'Yes').mean() * 100).sort_values()
    rate.plot(kind='barh', ax=ax, title=f'Attrition rate (%) by {col}')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, ['MonthlyIncome', 'YearsAtCompany', 'Age', 'DistanceFromHome']):
    sns.boxplot(data=raw, x='Attrition', y=col, ax=ax)
plt.tight_layout(); plt.show()

plt.figure(figsize=(14, 10))
sns.heatmap(raw.select_dtypes('number').corr(), cmap='coolwarm', center=0)
plt.title('Correlation heatmap'); plt.show()

**Write 4-5 insights here** (for example: about 16% of employees leave, so the data is imbalanced; overtime workers leave far more often; low income, short tenure and single status raise attrition).

## 4. Cleaning and feature engineering
`Gender` is removed from the model on purpose (fairness) but kept in `raw` for the audit later.

In [ ]:
DROP = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber', 'Gender']
y = raw['Attrition'].map({'Yes': 1, 'No': 0})
X_raw = raw.drop(columns=DROP + ['Attrition'])

def add_features(d):
    """Engineered features. Reused later by the simulator and the app so they never go stale."""
    d = d.copy()
    d['IncomePerYearExp'] = d['MonthlyIncome'] / (d['TotalWorkingYears'] + 1)
    d['TenureRatio']      = d['YearsAtCompany'] / (d['TotalWorkingYears'] + 1)
    d['PromotionGap']     = d['YearsSinceLastPromotion'] / (d['YearsAtCompany'] + 1)
    d['SatisfactionScore'] = d[['JobSatisfaction', 'EnvironmentSatisfaction',
                                'RelationshipSatisfaction', 'WorkLifeBalance']].mean(axis=1)
    return d

X = add_features(X_raw)
print(X.shape)

## 5. Train/test split and preprocessing
Split first, so nothing from the test set leaks into training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

num_cols = X.select_dtypes('number').columns.tolist()
cat_cols = X.select_dtypes('object').columns.tolist()

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])
print("Train:", X_train.shape, "| Test:", X_test.shape, "| Attrition rate:", round(y.mean(), 3))

## 6. Model comparison (5-fold stratified CV, training data only)
Imbalance is handled by class weights (and SMOTE for one comparison). Class weights keep probabilities easier to calibrate.

In [ ]:
spw = (y_train == 0).sum() / (y_train == 1).sum()
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

candidates = {
    'LogReg (class weights)': Pipeline([('prep', preprocess),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))]),
    'LogReg + SMOTE': Pipeline([('prep', preprocess), ('smote', SMOTE(random_state=SEED)),
        ('model', LogisticRegression(max_iter=2000))]),
    'RandomForest (class weights)': Pipeline([('prep', preprocess),
        ('model', RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                         class_weight='balanced_subsample',
                                         random_state=SEED, n_jobs=-1))]),
    'XGBoost (scale_pos_weight)': Pipeline([('prep', preprocess),
        ('model', XGBClassifier(scale_pos_weight=spw, eval_metric='logloss',
                                random_state=SEED, n_jobs=-1))]),
}

rows = {}
for name, pipe in candidates.items():
    s = cross_validate(pipe, X_train, y_train, cv=cv,
                       scoring=['roc_auc', 'average_precision', 'recall', 'precision', 'f1'])
    rows[name] = {k[5:]: v.mean() for k, v in s.items() if k.startswith('test_')}

cv_table = pd.DataFrame(rows).T.sort_values('roc_auc', ascending=False).round(3)
display(cv_table)

## 7. Hyperparameter tuning (XGBoost)

In [ ]:
xgb_pipe = Pipeline([('prep', preprocess),
                     ('model', XGBClassifier(scale_pos_weight=spw, eval_metric='logloss',
                                             random_state=SEED, n_jobs=1))])
param_dist = {
    'model__n_estimators':     [100, 200, 300, 400],
    'model__max_depth':        [2, 3, 4, 5],
    'model__learning_rate':    [0.01, 0.03, 0.05, 0.1],
    'model__subsample':        [0.7, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
    'model__min_child_weight': [1, 3, 5],
    'model__reg_lambda':       [1, 5, 10],
}
search = RandomizedSearchCV(xgb_pipe, param_dist, n_iter=30, scoring='roc_auc',
                            cv=cv, random_state=SEED, n_jobs=-1, verbose=1)
search.fit(X_train, y_train)
best = search.best_estimator_          # raw tuned pipeline (used for SHAP)
print("Best CV ROC-AUC:", round(search.best_score_, 3))
print(search.best_params_)

## 8. Calibration and cost-based threshold
* Calibration makes predicted probabilities honest (class weighting inflates them).
* The threshold is chosen on **out-of-fold training predictions**, never on the test set.
* It maximises estimated net savings using the business assumptions in Cell 2.

In [ ]:
def net_savings(y_true, y_pred):
    """Rs saved by acting on flagged employees, minus the cost of every intervention."""
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    return tp * (SUCCESS_RATE * COST_REPLACE - COST_INTERVENTION) - fp * COST_INTERVENTION

cal_model = CalibratedClassifierCV(best, method='sigmoid', cv=5).fit(X_train, y_train)

oof = cross_val_predict(CalibratedClassifierCV(best, method='sigmoid', cv=5),
                        X_train, y_train, cv=cv, method='predict_proba')[:, 1]

grid = np.arange(0.05, 0.90, 0.01)
y_tr = y_train.values
oof_savings = np.array([net_savings(y_tr, (oof >= g).astype(int)) for g in grid])
thr = float(grid[int(np.argmax(oof_savings))])

print(f"Break-even risk (theory): {COST_INTERVENTION / (SUCCESS_RATE * COST_REPLACE):.2f}")
print(f"Chosen threshold (data) : {thr:.2f}")

## 9. Evaluation on the untouched test set

In [ ]:
proba = cal_model.predict_proba(X_test)[:, 1]
pred = (proba >= thr).astype(int)
yt = y_test.values

print(f"ROC-AUC           : {roc_auc_score(yt, proba):.3f}")
print(f"PR-AUC            : {average_precision_score(yt, proba):.3f}  (baseline = {yt.mean():.3f})")
print(f"Brier score       : {brier_score_loss(yt, proba):.3f}  (lower is better)")
print(f"Flagged as leavers: {pred.mean():.1%} | Actual leavers: {yt.mean():.1%}\n")
print(classification_report(yt, pred, target_names=['Stay', 'Leave']))

s_model = net_savings(yt, pred)
s_all   = net_savings(yt, np.ones_like(yt))
s_best  = net_savings(yt, yt)
print(f"Estimated net savings on test set (Rs):")
print(f"  Using the model        : {s_model:,.0f}")
print(f"  Flag everyone          : {s_all:,.0f}")
print(f"  Do nothing             : 0")
print(f"  Perfect-knowledge cap  : {s_best:,.0f}")

fig, ax = plt.subplots(2, 2, figsize=(14, 11))
sns.heatmap(confusion_matrix(yt, pred), annot=True, fmt='d', cmap='Blues', ax=ax[0, 0],
            xticklabels=['Stay', 'Leave'], yticklabels=['Stay', 'Leave'])
ax[0, 0].set_title(f'Confusion matrix (threshold {thr:.2f})'); ax[0, 0].set_xlabel('Predicted'); ax[0, 0].set_ylabel('Actual')
RocCurveDisplay.from_predictions(yt, proba, ax=ax[0, 1]); ax[0, 1].set_title('ROC curve')
frac_pos, mean_pred = calibration_curve(yt, proba, n_bins=6, strategy='quantile')
ax[1, 0].plot(mean_pred, frac_pos, 'o-', label='Model'); ax[1, 0].plot([0, 1], [0, 1], '--', label='Perfect')
ax[1, 0].set_title('Calibration curve'); ax[1, 0].set_xlabel('Predicted risk'); ax[1, 0].set_ylabel('Actual leave rate'); ax[1, 0].legend()
ax[1, 1].plot(grid, oof_savings / 1e6); ax[1, 1].axvline(thr, color='red', ls='--', label=f'chosen {thr:.2f}')
ax[1, 1].set_title('Net savings vs threshold (training OOF)'); ax[1, 1].set_xlabel('Threshold'); ax[1, 1].set_ylabel('Rs million'); ax[1, 1].legend()
plt.tight_layout(); plt.savefig('evaluation.png', dpi=150); plt.show()

## 10. Explainability (SHAP values from the tuned XGBoost)
SHAP values are taken directly from XGBoost's built-in contributions (identical to TreeExplainer, but robust to version changes).

In [ ]:
prep_fit  = best.named_steps['prep']
booster   = best.named_steps['model'].get_booster()
feat_names = [n.split('__', 1)[1] for n in prep_fit.get_feature_names_out()]

Xt = prep_fit.transform(X_test)
contrib = booster.predict(xgb.DMatrix(Xt), pred_contribs=True)[:, :-1]   # drop bias column

shap.summary_plot(contrib, Xt, feature_names=feat_names, show=False)
plt.tight_layout(); plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight'); plt.show()

top = pd.Series(np.abs(contrib).mean(axis=0), index=feat_names).sort_values(ascending=False).head(10)
print("Top 10 global drivers (mean |SHAP|):"); display(top.round(3))

def top_drivers(raw_rows, k=3):
    """Main factors pushing ONE employee's risk up. raw_rows = one-row DataFrame in raw format."""
    Xr = add_features(raw_rows)[X_train.columns]
    c = booster.predict(xgb.DMatrix(prep_fit.transform(Xr)), pred_contribs=True)[0][:-1]
    idx = np.argsort(c)[::-1][:k]
    return [(feat_names[i], float(c[i])) for i in idx if c[i] > 0]

## 11. Fairness audit (responsible-AI check)
Gender was not a model input. We still check that outcomes are similar across groups.

In [ ]:
def audit(groups, title):
    d = pd.DataFrame({'group': np.asarray(groups), 'y': yt, 'pred': pred})
    out = []
    for g, part in d.groupby('group'):
        out.append({'group': g, 'n': len(part),
                    'actual_leave_rate': part['y'].mean(),
                    'flagged_rate': part['pred'].mean(),
                    'recall': part.loc[part['y'] == 1, 'pred'].mean(),
                    'precision': part.loc[part['pred'] == 1, 'y'].mean()})
    print(f"--- {title} ---"); display(pd.DataFrame(out).round(2))

audit(raw.loc[X_test.index, 'Gender'], 'By gender (not a model input)')
audit(pd.cut(X_test['Age'], [17, 30, 40, 70], labels=['<=30', '31-40', '41+']), 'By age group')

## 12. UNIQUE FEATURE: Retention What-If Simulator
For any employee the model re-scores the profile after each HR action and ranks actions by risk reduction.

**Honest caveat:** the model learned correlations from historical data, not proven causes. Treat the output as *model sensitivity to guide HR conversations*, not a guarantee.

In [ ]:
def risk(row):
    """Calibrated probability that this employee leaves. row = dict of RAW columns."""
    d = add_features(pd.DataFrame([row]))[X_train.columns]
    return float(cal_model.predict_proba(d)[0, 1])

ACTIONS = {
    "Eliminate overtime":              lambda r: {**r, 'OverTime': 'No'},
    "Add stock options (+1 level)":    lambda r: {**r, 'StockOptionLevel': min(3, r['StockOptionLevel'] + 1)},
    "15% salary raise":                lambda r: {**r, 'MonthlyIncome': r['MonthlyIncome'] * 1.15},
    "Improve work-life balance (+1)":  lambda r: {**r, 'WorkLifeBalance': min(4, r['WorkLifeBalance'] + 1)},
    "Improve job satisfaction (+1)":   lambda r: {**r, 'JobSatisfaction': min(4, r['JobSatisfaction'] + 1)},
    "Promotion (reset promotion gap)": lambda r: {**r, 'YearsSinceLastPromotion': 0},
    "Cut frequent travel":             lambda r: {**r, 'BusinessTravel': 'Travel_Rarely' if r['BusinessTravel'] == 'Travel_Frequently' else r['BusinessTravel']},
    "Reduce commute / remote option":  lambda r: {**r, 'DistanceFromHome': min(r['DistanceFromHome'], 5)},
}

def what_if(row):
    base = risk(row)
    out = [(name, base, risk(f(row))) for name, f in ACTIONS.items()]
    tbl = pd.DataFrame(out, columns=['Action', 'Current risk', 'Risk after action'])
    tbl['Risk reduction'] = tbl['Current risk'] - tbl['Risk after action']
    return base, tbl.sort_values('Risk reduction', ascending=False).reset_index(drop=True)

## 13. HR Action Report: score the workforce
The model was trained on 80% of these records, so treat this as a demonstration on historical data. In real use you would score *current* employees.

In [ ]:
risk_all = cal_model.predict_proba(X)[:, 1]
t_high = max(0.5, thr * 1.5)

report = pd.DataFrame({'EmployeeNumber': raw['EmployeeNumber'], 'Department': raw['Department'],
                       'JobRole': raw['JobRole'], 'Attrition': y, 'Risk': risk_all})
report['Tier'] = pd.cut(report['Risk'], bins=[-1, thr, t_high, 2], labels=['Low', 'Medium', 'High'])
print(report['Tier'].value_counts())

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
(report.groupby('Department')['Risk'].mean() * 100).sort_values().plot(kind='barh', ax=ax[0], title='Average attrition risk (%) by department')
pd.crosstab(report['Department'], report['Tier']).plot(kind='bar', stacked=True, ax=ax[1], title='Risk tiers by department')
plt.tight_layout(); plt.show()

# Top 15 at-risk CURRENT employees, with drivers and best action
active = report[report['Attrition'] == 0].sort_values('Risk', ascending=False).head(15)
rows = []
for i in active.index:
    r = X_raw.loc[i].to_dict()
    base, wi = what_if(r)
    drv = top_drivers(X_raw.loc[[i]])
    rows.append({'EmployeeNumber': int(raw.loc[i, 'EmployeeNumber']),
                 'Department': raw.loc[i, 'Department'], 'JobRole': raw.loc[i, 'JobRole'],
                 'Risk %': round(base * 100, 1),
                 'Main risk drivers': ', '.join(n for n, _ in drv),
                 'Best action': wi.loc[0, 'Action'],
                 'Risk after action %': round(wi.loc[0, 'Risk after action'] * 100, 1)})
action_report = pd.DataFrame(rows)
display(action_report)
action_report.to_csv('hr_action_report.csv', index=False)

## 14. Interactive app (Gradio)
Rebuilds a full employee profile from the inputs, applies the tuned threshold, shows the main risk drivers and runs the What-If Simulator.

In [ ]:
import gradio as gr

num_raw = X_raw.select_dtypes('number').columns
defaults = {c: (float(X_raw[c].median()) if c in num_raw else X_raw[c].mode()[0]) for c in X_raw.columns}
role_dept = X_raw.groupby('JobRole')['Department'].agg(lambda s: s.mode()[0]).to_dict()

def build_row(age, income, overtime, years_company, total_years, job_sat, env_sat, wlb, rel_sat,
              distance, stock, marital, travel, role, since_promo, job_level):
    total_years = max(total_years, years_company)
    total_years = min(total_years, max(age - 18, 0))
    years_company = min(years_company, total_years)
    r = dict(defaults)
    r.update(Age=age, MonthlyIncome=income, OverTime=overtime, YearsAtCompany=years_company,
             TotalWorkingYears=total_years, JobSatisfaction=job_sat, EnvironmentSatisfaction=env_sat,
             WorkLifeBalance=wlb, RelationshipSatisfaction=rel_sat, DistanceFromHome=distance,
             StockOptionLevel=stock, MaritalStatus=marital, BusinessTravel=travel, JobRole=role,
             JobLevel=job_level, Department=role_dept[role],
             YearsSinceLastPromotion=min(since_promo, years_company),
             YearsInCurrentRole=min(defaults['YearsInCurrentRole'], years_company),
             YearsWithCurrManager=min(defaults['YearsWithCurrManager'], years_company))
    return r

def app_predict(*inputs):
    row = build_row(*inputs)
    base, wi = what_if(row)
    pct = (risk_all < base).mean() * 100
    drv = top_drivers(pd.DataFrame([row]))
    if base >= t_high:   verdict = "HIGH RISK: likely to leave"
    elif base >= thr:    verdict = "MEDIUM RISK: likely to leave, act soon"
    else:                verdict = "LOW RISK: likely to stay"
    summary = (f"{verdict}\n"
               f"Attrition probability: {base * 100:.1f}%\n"
               f"Action threshold: {thr * 100:.0f}%  (cost-optimised)\n"
               f"Riskier than {pct:.0f}% of employees in the dataset\n"
               f"Main factors pushing risk up: {', '.join(n for n, _ in drv) or 'none'}")
    table = wi.copy()
    for c in ['Current risk', 'Risk after action', 'Risk reduction']:
        table[c] = (table[c] * 100).round(1).astype(str) + '%'
    return summary, table

In [ ]:
# Sanity check: the model must clearly separate these two profiles
high = (24, 2500, 'Yes', 1, 2, 1, 1, 1, 2, 20, 0, 'Single', 'Travel_Frequently', 'Sales Representative', 1, 1)
low  = (45, 12000, 'No', 12, 20, 4, 4, 3, 4, 3, 2, 'Married', 'Non-Travel', 'Manager', 2, 4)
print("HIGH-RISK PROFILE\n", app_predict(*high)[0], "\n")
print("LOW-RISK PROFILE\n", app_predict(*low)[0])

In [ ]:
demo = gr.Interface(
    fn=app_predict,
    inputs=[
        gr.Slider(18, 60, value=30, step=1, label="Age"),
        gr.Number(value=3000, label="Monthly Income"),
        gr.Radio(["Yes", "No"], value="No", label="OverTime"),
        gr.Slider(0, 40, value=2, step=1, label="Years at Company"),
        gr.Slider(0, 40, value=5, step=1, label="Total Working Years"),
        gr.Slider(1, 4, value=3, step=1, label="Job Satisfaction"),
        gr.Slider(1, 4, value=3, step=1, label="Environment Satisfaction"),
        gr.Slider(1, 4, value=3, step=1, label="Work-Life Balance"),
        gr.Slider(1, 4, value=3, step=1, label="Relationship Satisfaction"),
        gr.Slider(1, 30, value=10, step=1, label="Distance From Home"),
        gr.Slider(0, 3, value=1, step=1, label="Stock Option Level"),
        gr.Dropdown(sorted(X_raw['MaritalStatus'].unique()), value="Single", label="Marital Status"),
        gr.Dropdown(sorted(X_raw['BusinessTravel'].unique()), value="Travel_Rarely", label="Business Travel"),
        gr.Dropdown(sorted(X_raw['JobRole'].unique()), value="Sales Representative", label="Job Role"),
        gr.Slider(0, 15, value=1, step=1, label="Years Since Last Promotion"),
        gr.Slider(1, 5, value=1, step=1, label="Job Level"),
    ],
    outputs=[gr.Textbox(label="Risk assessment", lines=6),
             gr.Dataframe(label="Retention What-If Simulator (sorted by impact)")],
    title="Employee Attrition Predictor + Retention Simulator",
    description="Calibrated risk, cost-optimised threshold, main risk drivers and ranked HR actions.")
demo.launch(share=True)

## 15. Save and download everything for GitHub

In [ ]:
artifact = {
    'model': cal_model,            # calibrated model -> probabilities
    'explainer_model': best,       # raw tuned pipeline -> per-employee risk drivers
    'threshold': thr, 't_high': t_high,
    'columns': list(X_train.columns),
    'defaults': defaults, 'role_dept': role_dept,
    'risk_all': risk_all,          # workforce scores -> percentile ranking
    'choices': {c: sorted(X_raw[c].unique().tolist()) for c in ['MaritalStatus', 'BusinessTravel', 'JobRole']},
}
joblib.dump(artifact, 'attrition_model.pkl')

# Pin the exact versions used here, so the pickle loads cleanly on Hugging Face / your laptop
from importlib.metadata import version
pkgs = ['pandas', 'numpy', 'scikit-learn', 'imbalanced-learn', 'xgboost', 'joblib']
with open('requirements.txt', 'w') as f:
    f.write('\n'.join(f"{p}=={version(p)}" for p in pkgs) + '\n')
print("Gradio version (use as sdk_version in your Hugging Face Space card):", gr.__version__)

for f in ['attrition_model.pkl', 'requirements.txt', 'evaluation.png', 'shap_summary.png', 'hr_action_report.csv']:
    files.download(f)

## 16. Resume and interview notes

**Resume bullets (replace X, Y, Z with your printed results):**
* Built an end-to-end employee attrition model on 1,470 records (scikit-learn, XGBoost), reaching ROC-AUC X and PR-AUC Y on a held-out test set.
* Calibrated probabilities and chose the decision threshold by maximising estimated retention savings (about Rs Z on the test set vs flagging everyone).
* Added SHAP explanations, a fairness audit and a Retention What-If Simulator that ranks HR actions by modelled risk reduction; deployed as a Gradio app.

**Say honestly in interviews:**
* The dataset is synthetic and small, so metrics will be noisy.
* The cost numbers are assumptions; HR would supply real ones.
* The simulator shows model sensitivity, not proven causal effects.
* With real data you would add drift monitoring, privacy controls and a stricter fairness review (age is itself a protected attribute in many places).